# A.I Research Paper Intelligence System (NLP Semantic Search)
**Dataset:** `CShorten/ML-ArXiv-Papers` from HuggingFace

**Pipeline:** Load → Clean → Embed (MiniLM) → Index (FAISS) → Search

## Step 1: Install Dependencies

In [ ]:
!pip install datasets sentence-transformers faiss-cpu scikit-learn

## Step 2: Load Dataset

In [ ]:
from datasets import load_dataset
dataset = load_dataset("CShorten/ML-ArXiv-Papers", split='train')
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
    num_rows: 117592
})


## Step 3: Convert to DataFrame & Explore

In [ ]:
import pandas as pd
df = pd.DataFrame(dataset)
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
display(df[['title', 'abstract']].head())

Columns: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract']
Shape: (117592, 4)


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


## Step 4: Subset & Preprocess

In [ ]:
df = df.head(15000)
print("Working shape:", df.shape)
df['abstract'] = df['abstract'].fillna('')
df["paper_text"] = df["title"] + " " + df["abstract"]
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False).str.strip()
print("Null check:")
print(df.isnull().sum())
display(df[["paper_text"]].head())

Working shape: (15000, 4)
Null check:
Unnamed: 0.1    0
Unnamed: 0      0
title           0
abstract        0
paper_text      0
dtype: int64


,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


## Step 5: Load Sentence Transformer Model

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(type(model))

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


## Step 6: Test Single & Batch Embeddings

In [ ]:
sample_text = df["paper_text"].iloc[0]
embedding = model.encode(sample_text)
print("Type:", type(embedding))
print("Shape:", embedding.shape)
print("First 56 values:", embedding[:56])

Type: <class 'numpy.ndarray'>
Shape: (384,)
First 56 values: [-0.13156408 -0.00678268 -0.00367609  0.03265162  0.11219638  0.01227266
  0.09816723 -0.09005228  0.04231165 -0.0197735  -0.03308417  0.07452939
  0.10632038 -0.0206043  -0.02052102  0.00169492  0.07081947  0.05854449
 -0.1123191   0.02082469  0.05692545  0.0201578   0.025831    0.03217029
  0.10513768 -0.0967676   0.02700802 -0.0234509  -0.04549679 -0.01013698
 -0.01794855 -0.04814427  0.01077653 -0.03759069  0.01943478  0.0371519
  0.02967843  0.04330944  0.04373208  0.03704866 -0.00182593  0.00455186
 -0.00799067  0.03037373 -0.014378    0.0379515   0.05959161 -0.02583361
 -0.0652157   0.05900266 -0.02107138  0.07359428 -0.05720108  0.0029406
  0.00767515 -0.03334165]


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
sample_embedding = model.encode(df["paper_text"].head(5).to_list())
print("Batch shape:", sample_embedding.shape)
for i in range(1, 5):
    sim = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[i].reshape(1, -1))
    print(f"Similarity [0] vs [{i}]:", sim)

Batch shape: (5, 384)
Similarity [0] vs [1]: [[0.36625272]]
Similarity [0] vs [2]: [[0.3352284]]
Similarity [0] vs [3]: [[0.15505117]]
Similarity [0] vs [4]: [[0.37421536]]


## Step 7: Generate & Save Full Embeddings (with Cache)

In [ ]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings...")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings...")
    embeddings = model.encode(df["paper_text"].tolist(), batch_size=32, show_progress_bar=True)
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")
print("Embeddings shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

Loading saved embeddings...
Embeddings shape: (15000, 384)
Dtype: float32


## Step 8: Build & Save FAISS Index (with Cache)

In [ ]:
import faiss
if os.path.exists("paper_faiss.index"):
    print("Loading saved FAISS index...")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index...")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")
    print("Index saved successfully!")
print("Total vectors in index:", index.ntotal)

Loading saved FAISS index...
Total vectors in index: 15000


In [ ]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


In [ ]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [ ]:
print(df.iloc[11873]["title"])

Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection


In [ ]:
def search_paper(query, k=5):
    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    return D,I

In [ ]:
D,I=search_paper("deep learning for medical image analysis")
print(D)
print(I)

[[0.6807246  0.67092216 0.6521999  0.62811756 0.6131154 ]]
[[10466 13730 11873 12691 11282]]


## Step 9: Search Function & Test Query

In [ ]:
def search_paper(query, k=5):
    query_embedding = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print(f"Similarity Score: {score:.4f}")
        print(f"Title: {df.iloc[idx]['title']}")
        print(f"Abstract: {df.iloc[idx]['abstract'][:500]}")
        print("-" * 60)
search_paper("deep learning for medical image analysis")

Similarity Score: 0.6807
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
------------------------------------------------------------
Similarity Score: 0.6709
Title: Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract:   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-

In [ ]:
!pip install transformers==4.46.3

In [ ]:
from transformers import pipeline

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [ ]:
from transformers import pipeline
summarizer=pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [ ]:
summary=summarizer(df.iloc[10466]["abstract"],max_length=120,min_length=40)
print(summary)

[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]


In [ ]:
type(summary)

list

In [ ]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [ ]:
for score, idx in zip(D[0], I[0]):
        print(f"Similarity Score: {score:.4f}")
        print(f"Title: {df.iloc[idx]['title']}")
        print(f"Abstract: {df.iloc[idx]['abstract'][:500]}")
        summary=summarizer(df.iloc[idx]["abstract"],max_length=120,min_length=40)
        print(summary)
        print(summary[0]["summary_text"])
        print()

Similarity Score: 0.6807
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
[{'summary_text': 'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'}]
The combination of tomographic imaging and deep learning, or mac

In [ ]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity Score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()
        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )
        print("Summary:")
        print(summary[0]["summary_text"])
        print("-" * 80)

In [ ]:
search_and_summarize("Deep learning in medical imaging",k=5)

Similarity Score: 0.73549867
Title: A Perspective on Deep Imaging
Abstract:   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Summary:
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.
------------------------------------------------------------------------

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Summary:
The study used a set of high-resolution images to test the accuracy of the neural network. It found that the network performed better than expected when it was given a large set of images to work with.
--------------------------------------------------------------------------------
Similarity Score: 0.61512965
Title: Deep Learning Microscopy
Abstract:   We demonstrate that a deep neural network can significantly improve optical
microscopy, enhancing its spatial resolution over a large field-of-view and
depth-of-field. After its training, the only input to this network is an image
acquired using a regular optical microscope, without any changes to its design.
We blindly tested this deep learning approach using various tissue samples that
are imaged with low-resolution and wide-field systems, where the network
rapidly outputs an image with rema

Summary:
We demonstrate that a deep neural network can significantly improve opticalmicroscopy. After its training, the only input to t

In [ ]:
!pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.1 MB/s eta 0:00:00


In [ ]:
from keybert import KeyBERT

In [ ]:
kw_model = KeyBERT(model)

In [ ]:
type(kw_model)

keybert._model.KeyBERT

In [ ]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [ ]:
text=df.iloc[10466]["abstract"]
keyword=kw_model.extract_keywords(text)
print(keyword)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [ ]:
print(type(keyword))

<class 'list'>


In [ ]:
print(type(keyword[0]))

<class 'tuple'>


In [ ]:
keyword=kw_model.extract_keywords(text,keyphrase_ngram_range=(1,3),stop_words="english")

In [ ]:
print(keyword)

[('tomographic imaging deep', 0.6704), ('imaging deep learning', 0.6543), ('learning medical imaging', 0.6041), ('imaging deep', 0.5919), ('medical imaging', 0.5281)]


In [ ]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity Score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()
        summary = summarizer(
            df.iloc[idx]["abstract"],
            max_length=120,
            min_length=40,
            do_sample=False
        )
        print("Summary:")
        print(summary[0]["summary_text"])
        print()
        keyword = kw_model.extract_keywords(
            df.iloc[idx]["abstract"], keyphrase_ngram_range=(1,3), stop_words="english"
        )
        print("Keywords:")
        for kw, s in keyword:
          print(kw)


---
# Enhancements




In [ ]:

!pip install -q requests networkx


In [ ]:
import requests
import re
import time
import numpy as np
import networkx as nx
from collections import Counter

assert 'df' in globals(), "Run the earlier cells first (df not found)"
assert 'model' in globals(), "Run the earlier cells first (model not found)"
assert 'index' in globals(), "Run the earlier cells first (FAISS index not found)"
print("Enhancement layer ready. df shape:", df.shape)


Enhancement layer ready. df shape: (15000, 5)


## Deep Research Mode

Output: **Common Findings, Research Gap, Future Work, Best Paper**


In [ ]:
def _dedupe_by_topic(idx_list, score_list, sim_threshold=0.92, top_k=20):
    """Remove near-duplicate / same-topic papers using embedding similarity
    so the 20 results aren't all variations of the same paper."""
    kept_idx, kept_score = [], []
    kept_embeddings = []
    for idx, score in zip(idx_list, score_list):
        if len(kept_idx) >= top_k:
            break
        emb = embeddings[idx].reshape(1, -1)
        is_dup = False
        for ke in kept_embeddings:
            sim = float(np.dot(emb, ke.T))
            if sim >= sim_threshold:
                is_dup = True
                break
        if not is_dup:
            kept_idx.append(idx)
            kept_score.append(score)
            kept_embeddings.append(emb)
    return kept_idx, kept_score


def deep_research(query, fetch_k=40, top_k=20, max_for_summary=6):
    """Deep Research Mode: pulls top `fetch_k` candidates from FAISS, removes
    duplicate-topic papers down to `top_k`, then produces a comparative
    synthesis (common findings) and surfaces the best-matching paper.

    Note: Research Gap / Future Work are NOT included here — extracting
    those reliably needs real reasoning over full paper text, which a
    summarization model (BART) and keyword regex cannot do honestly.
    That part needs an actual LLM call and is left out for now."""
    print(f" Deep Research Mode for: '{query}'\n")
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, fetch_k)
    kept_idx, kept_score = _dedupe_by_topic(I[0], D[0], sim_threshold=0.92, top_k=top_k)
    print(f"Collected {len(kept_idx)} distinct-topic papers (from {fetch_k} candidates)\n")
    papers = []
    for rank, (idx, score) in enumerate(zip(kept_idx, kept_score), start=1):
        row = df.iloc[idx]
        papers.append({
            "rank": rank,
            "idx": int(idx),
            "score": float(score),
            "title": row["title"],
            "abstract": row["abstract"],
        })
    top_subset = papers[:max_for_summary]
    mini_summaries = []
    for p in top_subset:
        text = p["abstract"]
        if len(text.split()) > 30:
            try:
                s = summarizer(text, max_length=60, min_length=20, do_sample=False)
                mini_summaries.append(f"- {p['title']}: {s[0]['summary_text']}")
            except Exception as e:
                mini_summaries.append(f"- {p['title']}: {text[:200]}")
        else:
            mini_summaries.append(f"- {p['title']}: {text}")

    combined_text = " ".join(mini_summaries)
    try:
        comparative = summarizer(combined_text, max_length=130, min_length=40, do_sample=False)[0]["summary_text"]
    except Exception:
        comparative = combined_text[:500]
    best_paper = max(papers, key=lambda p: p["score"]) if papers else None
    print("=" * 80)
    print("COMPARATIVE SUMMARY ACROSS TOP PAPERS")
    print("=" * 80)
    print(comparative)
    print()
    print("COMMON FINDINGS (per-paper summaries)")
    for m in mini_summaries:
        print(m)
    print()
    print("BEST PAPER (highest similarity to query)")
    if best_paper:
        print(f"Title: {best_paper['title']}")
        print(f"Similarity Score: {best_paper['score']:.4f}")
        print(f"Abstract: {best_paper['abstract'][:300]}...")
    print("=" * 80)
    return {
        "papers": papers,
        "comparative_summary": comparative,
        "best_paper": best_paper,
    }


In [ ]:
research_result = deep_research("deep learning for medical image analysis", fetch_k=40, top_k=20)


 Deep Research Mode for: 'deep learning for medical image analysis'

Collected 20 distinct-topic papers (from 40 candidates)



/tmp/ipykernel_3550/1768947903.py:12: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  sim = float(np.dot(emb, ke.T))


COMPARATIVE SUMMARY ACROSS TOP PAPERS
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance.

COMMON FINDINGS (per-paper summaries)
- A Perspective on Deep Imaging: The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.
- Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?: Training a deep convolutional neural network from scratch is difficult because it requires a large amount of labeled training data. A promis

## Web Search Integration

If FAISS top similarity score is `< 0.60`, automatically fall back to live web search
(Semantic Scholar API → arXiv API) and combine results with local results.


In [ ]:
SIMILARITY_THRESHOLD = 0.60
def search_semantic_scholar(query, limit=5):
    """Query the Semantic Scholar Graph API (no key required for light usage)."""
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    params = {
        "query": query,
        "limit": limit,
        "fields": "title,abstract,year,authors,url,citationCount",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        results = []
        for d in data:
            results.append({
                "source": "Semantic Scholar",
                "title": d.get("title"),
                "abstract": d.get("abstract") or "",
                "year": d.get("year"),
                "authors": [a.get("name") for a in d.get("authors", [])],
                "url": d.get("url"),
                "citationCount": d.get("citationCount"),
            })
        return results
    except Exception as e:
        print("Semantic Scholar request failed:", e)
        return []


def search_arxiv(query, limit=5):
    """Query the arXiv public API (Atom XML) as a second fallback source."""
    import xml.etree.ElementTree as ET
    url = "http://export.arxiv.org/api/query"
    params = {"search_query": f"all:{query}", "start": 0, "max_results": limit}
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        root = ET.fromstring(resp.content)
        ns = {"atom": "http://www.w3.org/2005/Atom"}
        results = []
        for entry in root.findall("atom:entry", ns):
            title = entry.find("atom:title", ns)
            summary = entry.find("atom:summary", ns)
            link = entry.find("atom:id", ns)
            authors = [a.find("atom:name", ns).text for a in entry.findall("atom:author", ns)]
            results.append({
                "source": "arXiv",
                "title": title.text.strip() if title is not None else "",
                "abstract": summary.text.strip() if summary is not None else "",
                "year": None,
                "authors": authors,
                "url": link.text if link is not None else None,
                "citationCount": None,
            })
        return results
    except Exception as e:
        print("arXiv request failed:", e)
        return []


def smart_search(query, k=5, sim_threshold=SIMILARITY_THRESHOLD):
    """Search local FAISS index first; if best similarity is below threshold,
    automatically combine with live Semantic Scholar + arXiv web search."""
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    local_results = []
    for score, idx in zip(D[0], I[0]):
        local_results.append({
            "source": "Local FAISS",
            "title": df.iloc[idx]["title"],
            "abstract": df.iloc[idx]["abstract"],
            "score": float(score),
        })

    top_score = local_results[0]["score"] if local_results else 0.0
    print(f"Top local similarity: {top_score:.4f}")

    web_results = []
    if top_score < sim_threshold:
        print(f"  Similarity below {sim_threshold} — falling back to web search...\n")
        web_results.extend(search_semantic_scholar(query, limit=k))
        web_results.extend(search_arxiv(query, limit=k))
    else:
        print(" Local results are strong enough, skipping web search.\n")

    print("=" * 80)
    print(f"LOCAL RESULTS ({len(local_results)})")
    print("=" * 80)
    for r in local_results:
        print(f"[{r['score']:.4f}] {r['title']}")

    if web_results:
        print()
        print("=" * 80)
        print(f"WEB RESULTS ({len(web_results)})")
        print("=" * 80)
        for r in web_results:
            print(f"[{r['source']}] {r['title']} ({r.get('year', 'n/a')})")

    return {"local": local_results, "web": web_results}


In [ ]:
_ = smart_search("transformer attention mechanism", k=5)

print()
_ = smart_search("latest LLM agent benchmarks 2025", k=5)


Top local similarity: 0.4685
  Similarity below 0.6 — falling back to web search...

Semantic Scholar request failed: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=transformer+attention+mechanism&limit=5&fields=title%2Cabstract%2Cyear%2Cauthors%2Curl%2CcitationCount
LOCAL RESULTS (5)
[0.4685] Radio Transformer Networks: Attention Models for Learning to Synchronize
  in Wireless Systems
[0.4527] The Attentive Perceptron
[0.4503] A Controller-Recognizer Framework: How necessary is recognition for
  control?
[0.4491] Emergence of foveal image sampling from learning to attend in visual
  scenes
[0.4269] Sensor Transformation Attention Networks

WEB RESULTS (5)
[arXiv] Transformer-based Personalized Attention Mechanism for Medical Images with Clinical Records (None)
[arXiv] Dilated Neighborhood Attention Transformer (None)
[arXiv] Music Transformer (None)
[arXiv] Déjà vu: A Contextualized Temporal Attention Mechanism for Sequential Recommendation (N

## NER — Named Entity Recognition

Automatically extract **Models, Algorithms, Datasets, Libraries, Organizations, Authors**
mentioned in a paper's abstract.


In [ ]:
from transformers import pipeline as _pipeline
ner_pipeline = _pipeline("ner", grouped_entities=True)
KNOWN_MODELS = ["BERT", "RoBERTa", "GPT", "GPT-2", "GPT-3", "GPT-4", "T5", "BART",
                 "Transformer", "ResNet", "VGG", "LSTM", "GRU", "CNN", "RNN", "ViT",
                 "AlexNet", "Inception", "YOLO", "U-Net", "XLNet", "ELECTRA", "ALBERT"]
KNOWN_DATASETS = ["ImageNet", "MNIST", "CIFAR-10", "CIFAR-100", "COCO", "SQuAD",
                   "GLUE", "WikiText", "Penn Treebank", "CelebA", "Pascal VOC", "Wikipedia"]
KNOWN_LIBRARIES = ["PyTorch", "TensorFlow", "Keras", "scikit-learn", "Hugging Face",
                    "spaCy", "NLTK", "JAX", "FAISS", "OpenCV"]
KNOWN_ALGORITHMS = ["SGD", "Adam", "AdamW", "Backpropagation", "Gradient Descent",
                     "Reinforcement Learning", "Q-Learning", "K-Means", "Random Forest",
                     "SVM", "Decision Tree", "Bayesian Optimization"]

def extract_entities(text):
    """Extract Models, Algorithms, Datasets, Libraries, Organizations from a paper's text
    by combining a pretrained NER pipeline with domain-specific keyword matching
    (generic NER models are not trained on ML jargon like 'BERT' or 'PyTorch')."""
    found = {"Models": set(), "Datasets": set(), "Libraries": set(),
             "Algorithms": set(), "Organizations": set(), "Misc_Entities": []}

    lower_text = text.lower()
    for m in KNOWN_MODELS:
        if m.lower() in lower_text:
            found["Models"].add(m)
    for d in KNOWN_DATASETS:
        if d.lower() in lower_text:
            found["Datasets"].add(d)
    for l in KNOWN_LIBRARIES:
        if l.lower() in lower_text:
            found["Libraries"].add(l)
    for a in KNOWN_ALGORITHMS:
        if a.lower() in lower_text:
            found["Algorithms"].add(a)

    try:
        ner_results = ner_pipeline(text[:512])
        for ent in ner_results:
            if ent["entity_group"] == "ORG":
                found["Organizations"].add(ent["word"])
            elif ent["entity_group"] in ("PER", "MISC", "LOC"):
                found["Misc_Entities"].append((ent["entity_group"], ent["word"]))
    except Exception as e:
        print("NER pipeline error:", e)
    return {k: (sorted(v) if isinstance(v, set) else v) for k, v in found.items()}


No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/token_classification.py:170: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


In [ ]:
sample_idx = 10466
sample_text = df.iloc[sample_idx]["abstract"]
print("Title:", df.iloc[sample_idx]["title"])
print()
entities = extract_entities(sample_text)
for category, items in entities.items():
    print(f"{category}: {items}")


Title: A Perspective on Deep Imaging

Models: []
Datasets: []
Libraries: []
Algorithms: []
Organizations: []
Misc_Entities: []


## Citation Graph
- **Similar Papers** (from local FAISS embeddings — always available)
- **Cited By / References** (from Semantic Scholar API — when the paper is found online)


In [ ]:
def get_similar_papers_local(paper_idx, k=5):
    """Use the paper's own embedding to find similar papers in the local FAISS index."""
    paper_embedding = embeddings[paper_idx].reshape(1, -1).astype("float32").copy()
    faiss.normalize_L2(paper_embedding)
    D, I = index.search(paper_embedding, k + 1)
    similar = []
    for score, idx in zip(D[0], I[0]):
        if idx == paper_idx:
            continue
        similar.append({"idx": int(idx), "title": df.iloc[idx]["title"], "score": float(score)})
    return similar[:k]


def get_citations_semantic_scholar(title, limit=5):
    """Look up a paper on Semantic Scholar by title and fetch its citations/references."""
    search_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    try:
        resp = requests.get(search_url, params={"query": title, "limit": 1, "fields": "paperId,title"}, timeout=10)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        if not data:
            return None
        paper_id = data[0]["paperId"]

        detail_url = f"https://api.semanticscholar.org/graph/v1/paper/{paper_id}"
        resp = requests.get(detail_url, params={
            "fields": f"title,citations.title,references.title,citationCount"
        }, timeout=10)
        resp.raise_for_status()
        detail = resp.json()
        return {
            "title": detail.get("title"),
            "citationCount": detail.get("citationCount"),
            "citedBy": [c.get("title") for c in (detail.get("citations") or [])[:limit]],
            "references": [r.get("title") for r in (detail.get("references") or [])[:limit]],
        }
    except Exception as e:
        print("Semantic Scholar citation lookup failed:", e)
        return None


def build_citation_graph(paper_idx, k_similar=5, k_citations=5):
    """Build a knowledge-graph-style view of a paper: similar papers (local) +
    citedBy/references (Semantic Scholar, if found online)."""
    title = df.iloc[paper_idx]["title"]
    G = nx.DiGraph()
    G.add_node(title, type="root")
    similar = get_similar_papers_local(paper_idx, k=k_similar)
    for s in similar:
        G.add_node(s["title"], type="similar")
        G.add_edge(title, s["title"], relation="similar_to", weight=s["score"])
    sem_data = get_citations_semantic_scholar(title, limit=k_citations)
    if sem_data:
        for c in sem_data["citedBy"]:
            if c:
                G.add_node(c, type="cited_by")
                G.add_edge(c, title, relation="cites")
        for r in sem_data["references"]:
            if r:
                G.add_node(r, type="reference")
                G.add_edge(title, r, relation="references")

    print(f" Paper: {title}\n")
    print(f" Similar Papers (local, {len(similar)}):")
    for s in similar:
        print(f"   - [{s['score']:.3f}] {s['title']}")
    if sem_data:
        print(f"\n Cited By ({len(sem_data['citedBy'])}) [Semantic Scholar, citationCount={sem_data['citationCount']}]:")
        for c in sem_data["citedBy"]:
            print(f"   - {c}")
        print(f"\n References ({len(sem_data['references'])}):")
        for r in sem_data["references"]:
            print(f"   - {r}")
    else:
        print("\n(Paper not found on Semantic Scholar — showing local similarity graph only)")
    print(f"\n  Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return G


In [ ]:
citation_graph = build_citation_graph(10466, k_similar=5, k_citations=5)


Semantic Scholar citation lookup failed: 429 Client Error:  for url: https://api.semanticscholar.org/graph/v1/paper/search?query=A+Perspective+on+Deep+Imaging&limit=1&fields=paperId%2Ctitle
 Paper: A Perspective on Deep Imaging

 Similar Papers (local, 5):
   - [0.669] Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection
   - [0.657] Spatially-Adaptive Reconstruction in Computed Tomography using Neural
  Networks
   - [0.642] Deep Learning for Photoacoustic Tomography from Sparse Data
   - [0.619] Framing U-Net via Deep Convolutional Framelets: Application to
  Sparse-view CT
   - [0.619] High-Resolution Breast Cancer Screening with Multi-View Deep
  Convolutional Neural Networks

(Paper not found on Semantic Scholar — showing local similarity graph only)

  Graph: 6 nodes, 5 edges


## Keyword Intelligence

Already have `KeyBERT` for single-abstract keywords. This extends it to aggregate
keywords **across a set of search results** to surface research trends / hot topics.


In [ ]:
def keyword_intelligence(query, k=10, top_n_keywords=15):
    """Run a FAISS search, extract KeyBERT keywords from every result, then
    aggregate them into overall Top Keywords / Research Trends / Hot Topics."""
    query_embedding = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    all_keywords = []
    per_paper = []
    for score, idx in zip(D[0], I[0]):
        abstract = df.iloc[idx]["abstract"]
        kws = kw_model.extract_keywords(
            abstract, keyphrase_ngram_range=(1, 3), stop_words="english", top_n=5
        )
        per_paper.append({"title": df.iloc[idx]["title"], "score": float(score), "keywords": kws})
        all_keywords.extend([kw for kw, _ in kws])
    freq = Counter(all_keywords)
    top_keywords = freq.most_common(top_n_keywords)

    print(f"Keyword Intelligence for: '{query}' (across top {k} papers)\n")
    print("=" * 80)
    print("TOP KEYWORDS (by frequency across results)")
    print("=" * 80)
    for kw, count in top_keywords:
        print(f"  {kw:<40} x{count}")
    print()
    print("=" * 80)
    print("RESEARCH TRENDS / HOT TOPICS (keywords appearing in 2+ papers)")
    print("=" * 80)
    hot_topics = [kw for kw, count in top_keywords if count >= 2]
    if hot_topics:
        for kw in hot_topics:
            print(f"   {kw}")
    else:
        print("  (No keyword repeated across multiple papers — topic set is diverse)")
    print()
    print("=" * 80)
    print("PER-PAPER KEYWORDS")
    print("=" * 80)
    for p in per_paper:
        kw_str = ", ".join(kw for kw, _ in p["keywords"])
        print(f"[{p['score']:.3f}] {p['title']}")
        print(f"    Keywords: {kw_str}\n")

    return {"top_keywords": top_keywords, "hot_topics": hot_topics, "per_paper": per_paper}


In [ ]:
_ = keyword_intelligence("deep learning for medical image analysis", k=10)


🔑 Keyword Intelligence for: 'deep learning for medical image analysis' (across top 10 papers)

TOP KEYWORDS (by frequency across results)
  tomographic imaging deep                 x1
  imaging deep learning                    x1
  learning medical imaging                 x1
  imaging deep                             x1
  medical imaging                          x1
  trained cnns fine                        x1
  cnn adequate fine                        x1
  training deep cnn                        x1
  cnns sufficient fine                     x1
  cnn pre trained                          x1
  classification mri images                x1
  classification mri                       x1
  mri dataset substantial                  x1
  mri dataset                              x1
  advances deep learning                   x1

RESEARCH TRENDS / HOT TOPICS (keywords appearing in 2+ papers)
  (No keyword repeated across multiple papers — topic set is diverse)

PER-PAPER KEYWORDS
[0.681] A Perspect